In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from datetime import datetime

In [18]:
def preprocess(df_input, reference_date=None, is_train=True):
    df = df_input.copy()
    
    if reference_date is None:
        reference_date = pd.Timestamp.today()

    # ── 1. Drop ID / irrelevant columns ──────────────────────────────────
    drop_always = [
        'quote_id', 'vehicle_number_plate',
        'second_driver_birthdate',  # 100% null
        'second_driver_claim_free_years',  # 100% null
        'postal_code_houses_owned_by_rental_association_ratio',  # 100% null
    ]
    df.drop(columns=[c for c in drop_always if c in df.columns], inplace=True)

    # ── 2. Drop zero variance / useless ──────────────────────────────────
    drop_low_variance = [
        'usage', 'vehicle_is_taxi', 'vehicle_number_of_wheels',
        'is_driver_owner', 'vehicle_can_be_registered',
        'vehicle_is_marked_for_export', 'vehicle_model',
        'municipality', 'postal_code',
        'postal_code_houses_built_before_1945_ratio',
        'postal_code_houses_built_45_to_65_ratio',
        'postal_code_houses_built_65_to_75_ratio',
        'postal_code_houses_built_75_to_85_ratio',
        'postal_code_houses_built_85_to_95_ratio',
        'postal_code_houses_built_95_to_05_ratio',
        'postal_code_houses_built_05_to_15_ratio',
    ]
    df.drop(columns=[c for c in drop_low_variance if c in df.columns], inplace=True)

    # ── 3. Parse dates → numeric features ────────────────────────────────
    # Flag inspection before dropping
    if 'vehicle_inspection_report_date' in df.columns:
        df['has_inspection'] = df['vehicle_inspection_report_date'].notnull().astype(int)

    date_cols = [
        'vehicle_first_registration_date',
        'vehicle_country_first_registration_date',
        'vehicle_last_registration_date',
        'vehicle_inspection_report_date',
        'vehicle_inspection_expiry_date',
    ]
    for col in [c for c in date_cols if c in df.columns]:
        df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

    if 'vehicle_first_registration_date' in df.columns and \
       'vehicle_years_since_first_registration' not in df.columns:
        df['vehicle_years_since_first_registration'] = (
            (reference_date - df['vehicle_first_registration_date']).dt.days // 365
        )
    if 'vehicle_inspection_expiry_date' in df.columns:
        df['vehicle_days_until_inspection_expiry'] = (
            df['vehicle_inspection_expiry_date'] - reference_date
        ).dt.days.clip(lower=-3650)
    if 'vehicle_last_registration_date' in df.columns and \
       'vehicle_years_since_last_registration' not in df.columns:
        df['vehicle_years_since_last_registration'] = (
            (reference_date - df['vehicle_last_registration_date']).dt.days // 365
        )

    df.drop(columns=[c for c in date_cols if c in df.columns], inplace=True)

    # ── 4. Birthdate → age ───────────────────────────────────────────────
    if 'contractor_birthdate' in df.columns:
        df['contractor_birthdate'] = pd.to_datetime(
            df['contractor_birthdate'], dayfirst=True, errors='coerce'
        )
        df['contractor_age'] = (
            (reference_date - df['contractor_birthdate']).dt.days // 365
        )
        df.drop(columns=['contractor_birthdate'], inplace=True)

    # ── 5. Convert numeric strings to float ──────────────────────────────
    to_numeric_cols = [
        'vehicle_engine_size', 'vehicle_power', 'vehicle_net_weight',
        'vehicle_gross_weight', 'vehicle_length', 'vehicle_width',
        'vehicle_height', 'vehicle_net_max_power', 'vehicle_net_max_power_electric',
        'vehicle_nominal_continuous_max_power', 'vehicle_power_to_net_weight_ratio',
        'vehicle_number_of_cylinders', 'vehicle_number_of_doors', 'vehicle_number_of_seats',
        'vehicle_value_new', 'vehicle_planned_annual_mileage', 'vehicle_age',
        'vehicle_years_since_country_first_registration',
        'vehicle_year_of_last_odometer_report',
        'postal_code_latitude', 'postal_code_longitude', 'postal_code_distance_to_border',
        'postal_code_population', 'postal_code_households', 'postal_code_houses',
        'postal_code_average_household_size', 'postal_code_average_property_value',
        'postal_code_address_density', 'municipality_crimes_per_1000',
        'postal_code_0_to_15_years_inhabitants_ratio',
        'postal_code_25_to_45_years_inhabitants_ratio',
        'postal_code_45_to_65_years_inhabitants_ratio',
        'postal_code_65_years_and_older_inhabitants_ratio',
        'postal_code_single_person_households_ratio',
        'postal_code_multi_person_households_without_children_ratio',
        'postal_code_two_parent_households_ratio',
        'postal_code_social_benefit_recipients_ratio',
        'postal_code_owner_occupied_houses_ratio',
        'postal_code_rental_houses_ratio',
        'postal_code_multi_family_houses_ratio',
        'postal_code_nearest_train_station_distance',
        'postal_code_nearest_train_transfer_station_distance',
        'postal_code_nearest_motorway_ramp_distance',
        'postal_code_nearest_hospital_excl_clinic_distance',
        'postal_code_nearest_hospital_incl_clinic_distance',
        'postal_code_nearest_general_practitioner_distance',
        'postal_code_nearest_gp_post_service_distance',
        'postal_code_nearest_pharmacy_distance',
        'postal_code_nearest_primary_school_distance',
        'postal_code_nearest_secondary_school_distance',
        'postal_code_nearest_secondary_school_havo_vwo_distance',
        'postal_code_nearest_secondary_school_vmbo_distance',
        'postal_code_nearest_supermarket_distance',
        'postal_code_nearest_groceries_distance',
        'postal_code_nearest_restaurant_distance',
        'postal_code_nearest_cafe_distance',
        'postal_code_nearest_snackbar_distance',
        'postal_code_nearest_hotel_distance',
        'postal_code_nearest_library_distance',
        'postal_code_nearest_museum_distance',
        'postal_code_nearest_cinema_distance',
        'postal_code_nearest_department_store_distance',
        'postal_code_nearest_fire_station_distance',
        'postal_code_nearest_daycare_distance',
        'postal_code_nearest_after_school_care_distance',
        'postal_code_nearest_attraction_distance',
        'postal_code_nearest_performance_venue_distance',
        'postal_code_nearest_music_venue_distance',
        'postal_code_nearest_swimming_pool_distance',
        'postal_code_nearest_ice_rink_distance',
        'postal_code_nearest_sauna_distance',
        'postal_code_nearest_tanning_salon_distance',
        'postal_code_hospitals_excl_clinic_within_10_km',
        'postal_code_hospitals_incl_clinic_within_10_km',
        'postal_code_general_practitioners_within_3_km',
        'postal_code_primary_schools_within_3_km',
        'postal_code_secondary_schools_within_5_km',
        'postal_code_secondary_schools_havo_vwo_within_5_km',
        'postal_code_secondary_schools_vmbo_within_5_km',
        'postal_code_supermarkets_within_3_km',
        'postal_code_groceries_within_3_km',
        'postal_code_restaurants_within_3_km',
        'postal_code_cafes_within_3_km',
        'postal_code_snackbars_within_3_km',
        'postal_code_hotels_within_10_km',
        'postal_code_museums_within_10_km',
        'postal_code_cinemas_within_10_km',
        'postal_code_department_stores_within_10_km',
        'postal_code_daycares_within_3_km',
        'postal_code_after_school_cares_within_3_km',
        'postal_code_attractions_within_20_km',
        'postal_code_performance_venues_within_10_km',
        'vehicle_ownership_duration', 'claim_free_years',
        'postal_code_urban_category',
    ]
    for col in [c for c in to_numeric_cols if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # ── 6. Electric vehicle features ─────────────────────────────────────
    if 'vehicle_net_max_power_electric' in df.columns:
        df = df.copy()
        df['vehicle_is_electric'] = df['vehicle_net_max_power_electric'].notnull().astype(int)
        df['vehicle_net_max_power_electric'] = df['vehicle_net_max_power_electric'].fillna(0)
    if 'vehicle_nominal_continuous_max_power' in df.columns:
        df['vehicle_nominal_continuous_max_power'] = \
            df['vehicle_nominal_continuous_max_power'].fillna(0)

    # ── 7. Inspection status ──────────────────────────────────────────────
    if 'vehicle_inspection_number_of_deficiencies_found' in df.columns:
        df = df.copy()
        df['inspection_status'] = 0
        df.loc[df['vehicle_inspection_number_of_deficiencies_found'] == 1.0,
               'inspection_status'] = 1
        df.drop(columns=['vehicle_inspection_number_of_deficiencies_found',
                         'has_inspection'], inplace=True,
                errors='ignore')

    # ── 8. Binary mappings ────────────────────────────────────────────────
    binary_map = {'yes': 1, 'no': 0}
    binary_cols = [
        'vehicle_is_imported', 'vehicle_is_imported_within_last_12_months',
        'vehicle_has_open_recall',
    ]
    for col in [c for c in binary_cols if c in df.columns]:
        if df[col].dtype == object:
            df[col] = df[col].map(binary_map)
    df['vehicle_is_imported_within_last_12_months'] = \
        df['vehicle_is_imported_within_last_12_months'].fillna(0).astype(int)

    # ── 9. Payment frequency ──────────────────────────────────────────────
    if 'payment_frequency' in df.columns and df['payment_frequency'].dtype == object:
        df['payment_frequency'] = df['payment_frequency'].map(
            {'monthly': 0, 'yearly': 1}
        )

    df = df.copy()  # defragment
    return df



In [19]:

reference_date = pd.Timestamp('2026-01-01')  # fix date so both sets are consistent

df_raw = pd.read_parquet('../Hackathon2026Task-20260328T120731Z-3-001/Hackathon2026Task/block1_train.parquet')  # your actual filename
df_test_raw = pd.read_parquet('../Hackathon2026Task-20260328T120731Z-3-001/Hackathon2026Task/block2_test.parquet')  # your actual filename


df_train = preprocess(df_raw, reference_date=reference_date, is_train=True)
df_test  = preprocess(df_test_raw, reference_date=reference_date, is_train=False)

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")
print(f"\nTrain object cols: {df_train.select_dtypes(include='object').columns.tolist()}")
print(f"Test object cols:  {df_test.select_dtypes(include='object').columns.tolist()}")

Train shape: (541292, 133)
Test shape:  (164092, 122)

Train object cols: ['coverage', 'vehicle_maker', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_odometer_verdict_code', 'province']
Test object cols:  ['coverage', 'vehicle_maker', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_odometer_verdict_code', 'province']


In [20]:
train_not_test = set(df_train.columns) - set(df_test.columns)
test_not_train = set(df_test.columns) - set(df_train.columns)

print(f"In train but not test ({len(train_not_test)}):")
print(sorted(train_not_test))
print(f"\nIn test but not train ({len(test_not_train)}):")
print(sorted(test_not_train))

In train but not test (11):
['Insurer_A_price', 'Insurer_B_price', 'Insurer_C_price', 'Insurer_D_price', 'Insurer_E_price', 'Insurer_F_price', 'Insurer_G_price', 'Insurer_H_price', 'Insurer_I_price', 'Insurer_J_price', 'Insurer_K_price']

In test but not train (0):
[]


In [21]:
# Check correlation within postal code distance columns
dist_cols = [c for c in df_train.columns if 'nearest' in c]
size_cols = ['vehicle_length', 'vehicle_width', 'vehicle_height', 
             'vehicle_gross_weight', 'vehicle_net_weight']
ratio_cols = [c for c in df_train.columns if 'ratio' in c and 'postal' in c]
within_cols = [c for c in df_train.columns if 'within' in c]

print(f"Distance columns:    {len(dist_cols)}")
print(f"Vehicle size cols:   {len(size_cols)}")
print(f"Ratio columns:       {len(ratio_cols)}")
print(f"Within-radius cols:  {len(within_cols)}")
print(f"\nTotal reducible:     {len(dist_cols)+len(size_cols)+len(ratio_cols)+len(within_cols)}")
print(f"Current total cols:  {df_train.shape[1]}")

Distance columns:    32
Vehicle size cols:   5
Ratio columns:       11
Within-radius cols:  21

Total reducible:     69
Current total cols:  133


In [22]:
# Distance cols — keep only 3 most insurance-relevant
keep_distance = [
    'postal_code_nearest_motorway_ramp_distance',
    'postal_code_nearest_hospital_excl_clinic_distance',
    'postal_code_nearest_fire_station_distance',
]
drop_distance = [c for c in dist_cols if c not in keep_distance]

# Within-radius cols — keep only 3
keep_within = [
    'postal_code_hospitals_excl_clinic_within_10_km',
    'postal_code_supermarkets_within_3_km',
    'postal_code_restaurants_within_3_km',
]
drop_within = [c for c in within_cols if c not in keep_within]

# Vehicle size — keep only gross weight
drop_size = [c for c in size_cols if c != 'vehicle_gross_weight']

# Ratio cols — keep only 3
keep_ratio = [
    'postal_code_social_benefit_recipients_ratio',
    'postal_code_65_years_and_older_inhabitants_ratio',
    'postal_code_single_person_households_ratio',
]
drop_ratio = [c for c in ratio_cols if c not in keep_ratio]

# Other redundant cols
drop_other = [
    'vehicle_net_max_power_electric',
    'vehicle_nominal_continuous_max_power',
    'vehicle_years_since_country_first_registration',
    'vehicle_year_of_last_odometer_report',
    'postal_code_households',
    'postal_code_houses',
    'postal_code_average_household_size',
    'postal_code_latitude',
    'postal_code_longitude',
    'vehicle_days_until_inspection_expiry',  # 63% null, inspection_status covers this
    'vehicle_years_since_last_registration', # redundant with vehicle_age
]

# Combine all drops
all_drops = drop_distance + drop_within + drop_size + drop_ratio + drop_other
all_drops = [c for c in all_drops if c in df_train.columns]

print(f"Dropping {len(all_drops)} columns")
print(f"Remaining: {df_train.shape[1] - len(all_drops)} columns")

# Apply to both train and test
df_train.drop(columns=all_drops, inplace=True)
df_test.drop(columns=all_drops, inplace=True)

print(f"\nTrain shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

Dropping 70 columns
Remaining: 63 columns

Train shape: (541292, 63)
Test shape:  (164092, 52)


In [23]:
# # ── One-hot encode low cardinality cols ──────────────────────────────────
# onehot_cols = [
#     'vehicle_fuel_type',           # 7 values
#     'vehicle_primary_color',       # 15 values
#     'vehicle_odometer_verdict_code', # 8 values
#     'province',                    # 12 values
# ]

# # Combine train and test before encoding to ensure same columns appear in both
# df_train['_is_train'] = 1
# df_test['_is_train'] = 0
# combined = pd.concat([df_train, df_test], axis=0)

# combined = pd.get_dummies(combined, columns=onehot_cols, drop_first=False)

# # ── Target encode high cardinality cols ──────────────────────────────────
# # Compute mean price per maker from TRAIN only
# insurer_price_cols = [c for c in df_train.columns if c.endswith('_price')]
# train_only = combined[combined['_is_train'] == 1]

# maker_encoding = train_only.groupby('vehicle_maker')[insurer_price_cols].mean().mean(axis=1)
# maker_encoding.name = 'vehicle_maker_encoded'

# combined = combined.merge(maker_encoding, on='vehicle_maker', how='left')
# combined.drop(columns=['vehicle_maker'], inplace=True)

# # ── Split back into train and test ───────────────────────────────────────
# df_train = combined[combined['_is_train'] == 1].drop(columns=['_is_train'])
# df_test  = combined[combined['_is_train'] == 0].drop(columns=['_is_train', 
#             *insurer_price_cols])

# df_train = df_train.copy()
# df_test  = df_test.copy()

# print(f"Train shape: {df_train.shape}")
# print(f"Test shape:  {df_test.shape}")
# print(f"Remaining object cols in train: {df_train.select_dtypes(include='object').columns.tolist()}")
# print(f"Remaining object cols in test:  {df_test.select_dtypes(include='object').columns.tolist()}")

In [24]:
insurer_price_cols = [c for c in df_train.columns if c.endswith('_price')]

# Mean price per color across all insurers
color_price = df_train.groupby('vehicle_primary_color')[insurer_price_cols].mean()
color_price['avg_price'] = color_price.mean(axis=1)
print(color_price['avg_price'].sort_values(ascending=False).round(2))

# Also check how much variance color explains
overall_mean = df_train[insurer_price_cols].mean().mean()
color_std = color_price['avg_price'].std()
print(f"\nOverall mean price: {overall_mean:.2f}")
print(f"Std across colors:  {color_std:.2f}")
print(f"Max difference:     {color_price['avg_price'].max() - color_price['avg_price'].min():.2f}")

vehicle_primary_color
yellow            125.70
black             106.26
grey              102.41
blue              101.20
green             100.60
white              98.00
other              94.46
red                93.26
orange             92.31
not_registered     90.90
beige              89.85
brown              86.54
purple             85.68
cream              83.38
pink               83.01
Name: avg_price, dtype: float64

Overall mean price: 101.97
Std across colors:  10.98
Max difference:     42.69


In [25]:
# Check if color signal survives controlling for vehicle value
import numpy as np

# Bin vehicle value into quartiles
df_train['value_quartile'] = pd.qcut(df_train['vehicle_value_new'], q=4, labels=False)

# Mean price per color within each value quartile
color_controlled = df_train.groupby(['value_quartile', 'vehicle_primary_color'])[insurer_price_cols].mean()
color_controlled['avg_price'] = color_controlled.mean(axis=1)
color_std_controlled = color_controlled.groupby('vehicle_primary_color')['avg_price'].mean().std()

print(f"Std across colors (raw):        {10.98:.2f}")
print(f"Std across colors (controlled): {color_std_controlled:.2f}")

# Clean up temp column
df_train.drop(columns=['value_quartile'], inplace=True)

Std across colors (raw):        10.98
Std across colors (controlled): 8.37


In [26]:
insurer_price_cols = [c for c in df_train.columns if c.endswith('_price')]

# Mean price per province
province_price = df_train.groupby('province')[insurer_price_cols].mean()
province_price['avg_price'] = province_price.mean(axis=1)
print(province_price['avg_price'].sort_values(ascending=False).round(2))

overall_mean = df_train[insurer_price_cols].mean().mean()
province_std = province_price['avg_price'].std()
print(f"\nOverall mean price: {overall_mean:.2f}")
print(f"Std across provinces: {province_std:.2f}")
print(f"Max difference: {province_price['avg_price'].max() - province_price['avg_price'].min():.2f}")

province
Province_12    117.41
Province_8     108.00
Province_10    103.52
Province_2      98.31
Province_7      97.99
Province_6      97.78
Province_9      92.59
Province_5      92.58
Province_4      92.20
Province_11     84.86
Province_3      76.71
Province_1      76.64
Name: avg_price, dtype: float64

Overall mean price: 101.97
Std across provinces: 11.93
Max difference: 40.78


In [27]:
# Check correlation between province and the postal code features we kept
postal_proxy_cols = [
    'municipality_crimes_per_1000',
    'postal_code_urban_category',
    'postal_code_address_density',
    'postal_code_average_property_value',
    'postal_code_population',
]

print("\nMean postal code stats per province:")
print(df_train.groupby('province')[postal_proxy_cols].mean().round(3))


Mean postal code stats per province:
             municipality_crimes_per_1000  postal_code_urban_category  \
province                                                                
Province_1                          2.677                       3.927   
Province_10                         5.433                       2.378   
Province_11                         3.650                       3.739   
Province_12                         5.737                       1.972   
Province_2                          5.742                       2.983   
Province_3                          3.269                       3.667   
Province_4                          4.050                       3.133   
Province_5                          4.594                       2.786   
Province_6                          4.252                       3.197   
Province_7                          4.924                       2.883   
Province_8                          6.023                       2.126   
Province_9   

In [28]:
df_train.drop(columns=['province'], inplace=True)

df_test.drop(columns=['province'], inplace=True)

print(f"Train shape: {df_train.shape}")

print(f"Test shape:  {df_test.shape}")

print(f"Remaining object cols: {df_train.select_dtypes(include='object').columns.tolist()}")

Train shape: (541292, 62)
Test shape:  (164092, 51)
Remaining object cols: ['coverage', 'vehicle_maker', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_odometer_verdict_code']


In [30]:
insurer_price_cols = [c for c in df_train.columns if c.endswith('_price')]
df_train['avg_price'] = df_train[insurer_price_cols].mean(axis=1)

# Correlation of every numeric feature with avg price
numeric_cols = df_train.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in insurer_price_cols + ['avg_price']]

correlations = df_train[numeric_cols].corrwith(df_train['avg_price'])
correlations = correlations.abs().sort_values(ascending=False)

print("Feature correlation with avg price:")
print(correlations.round(3).to_string())

df_train.drop(columns=['avg_price'], inplace=True)

Feature correlation with avg price:
payment_frequency                                    0.633
contractor_age                                       0.234
claim_free_years                                     0.224
vehicle_value_new                                    0.178
vehicle_power                                        0.175
vehicle_gross_weight                                 0.162
vehicle_net_max_power                                0.162
Insurer_C_deductible                                 0.145
vehicle_is_electric                                  0.139
vehicle_engine_size                                  0.138
Insurer_H_deductible                                 0.135
vehicle_ownership_duration                           0.123
postal_code_hospitals_excl_clinic_within_10_km       0.121
postal_code_supermarkets_within_3_km                 0.116
postal_code_urban_category                           0.110
vehicle_number_of_cylinders                          0.104
Insurer_A_deductible

/home/petar/.local/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/petar/.local/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [31]:
print(df_train['inspection_status'].value_counts())
print(df_train['inspection_status'].dtype)

inspection_status
0    541292
Name: count, dtype: int64
int64


In [32]:
df_train.drop(columns=['inspection_status'], inplace=True)
df_test.drop(columns=['inspection_status'], inplace=True)

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

Train shape: (541292, 61)
Test shape:  (164092, 50)


In [33]:
cols_to_drop = [
    'postal_code_distance_to_border',
    'postal_code_average_property_value',
    'postal_code_nearest_fire_station_distance',
    'postal_code_single_person_households_ratio',
    'vehicle_is_imported',           # 0.050 — borderline
    'vehicle_planned_annual_mileage', # 0.045 — borderline  
    'vehicle_number_of_seats',        # 0.044 — borderline
    'vehicle_power_to_net_weight_ratio', # 0.039
    'vehicle_has_open_recall',        # 0.036
    'vehicle_number_of_doors',        # 0.033
    'postal_code_population',         # 0.029
    'postal_code_nearest_motorway_ramp_distance', # 0.029
    'municipality_crimes_per_1000',   # 0.027
]

cols_to_drop = [c for c in cols_to_drop if c in df_train.columns]
df_train.drop(columns=cols_to_drop, inplace=True)
df_test.drop(columns=cols_to_drop, inplace=True)

print(f"Dropped {len(cols_to_drop)} columns")
print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

Dropped 13 columns
Train shape: (541292, 48)
Test shape:  (164092, 37)


In [34]:
# Combine train and test for consistent encoding
df_train['_is_train'] = 1
df_test['_is_train'] = 0
combined = pd.concat([df_train, df_test], axis=0, ignore_index=True)

# ── One-hot encode ────────────────────────────────────────────────────────
onehot_cols = [
    'vehicle_fuel_type',
    'vehicle_primary_color',
    'vehicle_odometer_verdict_code',
]
combined = pd.get_dummies(combined, columns=onehot_cols, drop_first=False)

# ── Target encode vehicle_maker ───────────────────────────────────────────
insurer_price_cols = [c for c in df_train.columns if c.endswith('_price')]
train_only = combined[combined['_is_train'] == 1]

maker_encoding = train_only.groupby('vehicle_maker')[insurer_price_cols].mean().mean(axis=1)
maker_encoding.name = 'vehicle_maker_encoded'

combined = combined.merge(maker_encoding, on='vehicle_maker', how='left')
combined.drop(columns=['vehicle_maker'], inplace=True)

# ── Split back ────────────────────────────────────────────────────────────
df_train = combined[combined['_is_train'] == 1].drop(columns=['_is_train']).copy()
df_test  = combined[combined['_is_train'] == 0].drop(columns=['_is_train'] + insurer_price_cols).copy()

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")
print(f"Remaining object cols: {df_train.select_dtypes(include='object').columns.tolist()}")
print(f"\nSample of new one-hot columns:")
new_cols = [c for c in df_train.columns if any(c.startswith(p+'_') for p in onehot_cols)]
print(new_cols)

Train shape: (541292, 75)
Test shape:  (164092, 64)
Remaining object cols: ['coverage']

Sample of new one-hot columns:
['vehicle_fuel_type_diesel', 'vehicle_fuel_type_electric', 'vehicle_fuel_type_gas', 'vehicle_fuel_type_other', 'vehicle_fuel_type_petrol', 'vehicle_fuel_type_petrol_electric_hybrid', 'vehicle_fuel_type_petrol_gas', 'vehicle_primary_color_beige', 'vehicle_primary_color_black', 'vehicle_primary_color_blue', 'vehicle_primary_color_brown', 'vehicle_primary_color_cream', 'vehicle_primary_color_green', 'vehicle_primary_color_grey', 'vehicle_primary_color_not_registered', 'vehicle_primary_color_orange', 'vehicle_primary_color_other', 'vehicle_primary_color_pink', 'vehicle_primary_color_purple', 'vehicle_primary_color_red', 'vehicle_primary_color_white', 'vehicle_primary_color_yellow', 'vehicle_odometer_verdict_code_00', 'vehicle_odometer_verdict_code_01', 'vehicle_odometer_verdict_code_02', 'vehicle_odometer_verdict_code_03', 'vehicle_odometer_verdict_code_04', 'vehicle_odom

In [35]:
# Split by coverage
df_casco         = df_train[df_train['coverage'] == 'casco'].drop(columns=['coverage']).copy()
df_limited_casco = df_train[df_train['coverage'] == 'limited_casco'].drop(columns=['coverage']).copy()
df_mtpl          = df_train[df_train['coverage'] == 'mtpl'].drop(columns=['coverage']).copy()

# Test splits
df_test_casco         = df_test[df_test['coverage'] == 'casco'].drop(columns=['coverage']).copy()
df_test_limited_casco = df_test[df_test['coverage'] == 'limited_casco'].drop(columns=['coverage']).copy()
df_test_mtpl          = df_test[df_test['coverage'] == 'mtpl'].drop(columns=['coverage']).copy()

print(f"Train splits:")
print(f"  casco:         {df_casco.shape}")
print(f"  limited_casco: {df_limited_casco.shape}")
print(f"  mtpl:          {df_mtpl.shape}")
print(f"\nTest splits:")
print(f"  casco:         {df_test_casco.shape}")
print(f"  limited_casco: {df_test_limited_casco.shape}")
print(f"  mtpl:          {df_test_mtpl.shape}")

# Define feature columns (no price cols, no coverage)
price_cols = [c for c in df_train.columns if c.endswith('_price')]
feature_cols = [c for c in df_casco.columns if c not in price_cols]

print(f"\nFeature columns: {len(feature_cols)}")
print(f"Target columns:  {len(price_cols)}")


Train splits:
  casco:         (163742, 74)
  limited_casco: (238176, 74)
  mtpl:          (139374, 74)

Test splits:
  casco:         (51616, 63)
  limited_casco: (71589, 63)
  mtpl:          (40887, 63)

Feature columns: 63
Target columns:  11


In [ ]:
import numpy as np

LGBM_PARAMS = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'n_estimators': 3000,
    'learning_rate': 0.01,
    'num_leaves': 127,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbose': -1,
}

def train_coverage_models(df_train_cov, df_test_cov, coverage_name):
    models = {}
    predictions = {}
    scores = {}

    feature_cols = [c for c in df_train_cov.columns
                    if not c.endswith('_price')
                    and not c.endswith('_quoted')]

    for insurer in INSURER_COLS:
        mask = df_train_cov[insurer].notnull()
        X = df_train_cov.loc[mask, feature_cols]
        y = df_train_cov.loc[mask, insurer]

        if len(X) < 100:
            print(f"  {insurer}: skipping — only {len(X)} rows")
            continue

        y_log = np.log1p(y)

        X_train, X_val, y_train, y_val = train_test_split(
            X, y_log, test_size=0.1, random_state=42
        )

        model = lgb.LGBMRegressor(**LGBM_PARAMS)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(period=-1)]
        )

        val_pred = np.expm1(model.predict(X_val))
        val_actual = np.expm1(y_val)
        mae = mean_absolute_error(val_actual, val_pred)
        scores[insurer] = mae

        predictions[insurer] = np.expm1(model.predict(df_test_cov[feature_cols]))
        models[insurer] = model

        print(f"  {insurer}: MAE={mae:.2f}  trained on {len(X):,} rows  "
              f"best_iter={model.best_iteration_}")

    print(f"\n  Average MAE ({coverage_name}): "
          f"{sum(scores.values())/len(scores):.2f}")
    return models, predictions, scores

print("=== CASCO ===")
models_casco, preds_casco, scores_casco = train_coverage_models(
    df_casco, df_test_casco, 'casco'
)

print("\n=== LIMITED CASCO ===")
models_limited, preds_limited, scores_limited = train_coverage_models(
    df_limited_casco, df_test_limited_casco, 'limited_casco'
)

print("\n=== MTPL ===")
models_mtpl, preds_mtpl, scores_mtpl = train_coverage_models(
    df_mtpl, df_test_mtpl, 'mtpl'
)

all_scores = {**scores_casco, **scores_limited, **scores_mtpl}
print(f"\n=== OVERALL POOLED MAE: {sum(all_scores.values())/len(all_scores):.2f} ===")

=== CASCO ===
  Insurer_A_price: MAE=8.73  trained on 156,664 rows  best_iter=2000
  Insurer_B_price: MAE=9.01  trained on 140,963 rows  best_iter=2000
  Insurer_C_price: MAE=8.30  trained on 140,269 rows  best_iter=2000
  Insurer_D_price: MAE=10.16  trained on 134,654 rows  best_iter=2000
  Insurer_E_price: MAE=19.54  trained on 103,140 rows  best_iter=2000
  Insurer_F_price: MAE=10.76  trained on 126,022 rows  best_iter=2000
  Insurer_G_price: MAE=5.66  trained on 94,686 rows  best_iter=2000
  Insurer_H_price: MAE=13.95  trained on 7,308 rows  best_iter=2000
  Insurer_I_price: MAE=15.13  trained on 86,186 rows  best_iter=2000
  Insurer_J_price: MAE=10.30  trained on 120,809 rows  best_iter=2000
  Insurer_K_price: MAE=10.58  trained on 101,291 rows  best_iter=2000

  Average MAE (casco): 11.10

=== LIMITED CASCO ===
  Insurer_A_price: MAE=8.65  trained on 231,334 rows  best_iter=2000
  Insurer_B_price: MAE=7.63  trained on 186,607 rows  best_iter=2000
  Insurer_C_price: MAE=7.85  trai

In [39]:
for insurer in ['Insurer_E_price', 'Insurer_I_price', 'Insurer_K_price', 'Insurer_A_price']:
    vals = df_train[insurer].dropna()
    print(f"{insurer}: mean={vals.mean():.1f}  std={vals.std():.1f}  "
          f"min={vals.min():.1f}  max={vals.max():.1f}  skew={vals.skew():.2f}")

Insurer_E_price: mean=115.7  std=88.2  min=15.6  max=2283.8  skew=3.26
Insurer_I_price: mean=94.9  std=71.1  min=14.1  max=902.0  skew=2.29
Insurer_K_price: mean=124.0  std=83.6  min=15.7  max=1336.0  skew=1.79
Insurer_A_price: mean=96.9  std=115.5  min=12.3  max=4370.4  skew=8.89


In [40]:
for insurer in [c for c in df_train.columns if c.endswith('_price')]:
    vals = df_train[insurer].dropna()
    print(f"{insurer:25s} mean={vals.mean():7.1f}  std={vals.std():7.1f}  "
          f"skew={vals.skew():.2f}  max={vals.max():.1f}")

Insurer_A_price           mean=   96.9  std=  115.5  skew=8.89  max=4370.4
Insurer_B_price           mean=   99.8  std=   75.5  skew=2.51  max=1252.5
Insurer_C_price           mean=   89.3  std=   62.9  skew=2.40  max=1095.2
Insurer_D_price           mean=   98.5  std=   71.3  skew=2.12  max=858.6
Insurer_E_price           mean=  115.7  std=   88.2  skew=3.26  max=2283.8
Insurer_F_price           mean=   97.4  std=   66.1  skew=2.08  max=781.4
Insurer_G_price           mean=   84.5  std=   51.6  skew=1.73  max=646.5
Insurer_H_price           mean=  116.1  std=   76.0  skew=1.82  max=996.0
Insurer_I_price           mean=   94.9  std=   71.1  skew=2.29  max=902.0
Insurer_J_price           mean=  104.7  std=   74.2  skew=2.66  max=1451.8
Insurer_K_price           mean=  124.0  std=   83.6  skew=1.79  max=1336.0


In [42]:
print('quote_id' in df_test_raw.columns)
print(df_test_raw['quote_id'].head())

True
0    541293
1    541294
2    541295
3    541296
4    541297
Name: quote_id, dtype: int64


In [43]:
# Get quote_ids from raw test file
quote_ids = df_test_raw['quote_id'].values

# Coverage split indices — we need to know which quote_id belongs to which split
test_coverage = df_test_raw['coverage'].values

# Build empty submission with all quote_ids
submission = pd.DataFrame({'quote_id': quote_ids})

insurer_cols = [
    'Insurer_A_price', 'Insurer_B_price', 'Insurer_C_price',
    'Insurer_D_price', 'Insurer_E_price', 'Insurer_F_price',
    'Insurer_G_price', 'Insurer_H_price', 'Insurer_I_price',
    'Insurer_J_price', 'Insurer_K_price'
]

# Initialize all predictions as 0
for col in insurer_cols:
    submission[col] = 0.0

# Fill predictions per coverage split
for coverage, preds, test_split in [
    ('casco',         preds_casco,   df_test_casco),
    ('limited_casco', preds_limited, df_test_limited_casco),
    ('mtpl',          preds_mtpl,    df_test_mtpl),
]:
    # Get indices of this coverage in the original test set
    idx = df_test_raw[df_test_raw['coverage'] == coverage].index

    for insurer in insurer_cols:
        if insurer in preds:
            submission.loc[idx, insurer] = preds[insurer]

# Verify shape and no missing quote_ids
print(f"Submission shape: {submission.shape}")
print(f"Expected rows: {len(df_test_raw)}")
print(f"Any nulls: {submission.isnull().sum().sum()}")
print(f"\nSample:")
print(submission.head())

# Save with correct format — semicolon separator, dot decimal
submission.to_csv('submission.csv', sep=';', decimal='.', index=False)
print("\nSaved to submission.csv")

Submission shape: (164092, 12)
Expected rows: 164092
Any nulls: 0

Sample:
   quote_id  Insurer_A_price  Insurer_B_price  Insurer_C_price  \
0    541293        73.973576       109.699019        83.072877   
1    541294        30.387517        35.101336        31.992339   
2    541295        25.912147        33.128008        30.449078   
3    541296       322.811989       291.988676       344.845699   
4    541297       203.772888       290.138318       125.190533   

   Insurer_D_price  Insurer_E_price  Insurer_F_price  Insurer_G_price  \
0       110.541282       137.318678        75.973214        80.042106   
1        35.814282        43.654786        37.630029        31.230976   
2        28.294381        45.383679        29.850049        35.108015   
3       304.223049       484.852161       348.221136       252.094016   
4       304.624000       388.705816       189.297483       185.809102   

   Insurer_H_price  Insurer_I_price  Insurer_J_price  Insurer_K_price  
0        97.89609

In [44]:
# Read back and verify
check = pd.read_csv('submission.csv', sep=';')
print(f"Shape: {check.shape}")
print(f"Columns: {check.columns.tolist()}")
print(check.head(3).to_string())

Shape: (164092, 12)
Columns: ['quote_id', 'Insurer_A_price', 'Insurer_B_price', 'Insurer_C_price', 'Insurer_D_price', 'Insurer_E_price', 'Insurer_F_price', 'Insurer_G_price', 'Insurer_H_price', 'Insurer_I_price', 'Insurer_J_price', 'Insurer_K_price']
   quote_id  Insurer_A_price  Insurer_B_price  Insurer_C_price  Insurer_D_price  Insurer_E_price  Insurer_F_price  Insurer_G_price  Insurer_H_price  Insurer_I_price  Insurer_J_price  Insurer_K_price
0    541293        73.973576       109.699019        83.072877       110.541282       137.318678        75.973214        80.042106        97.896092       152.348007       198.839255       164.735663
1    541294        30.387517        35.101336        31.992339        35.814282        43.654786        37.630029        31.230976        69.462252        32.605420        44.462241        49.509800
2    541295        25.912147        33.128008        30.449078        28.294381        45.383679        29.850049        35.108015        32.590334     

In [45]:
# Check predictions are there and correct size
print("Casco predictions:")
for insurer, pred in preds_casco.items():
    print(f"  {insurer}: {len(pred)} rows (expected {len(df_test_casco)})")

print("\nLimited casco predictions:")
for insurer, pred in preds_limited.items():
    print(f"  {insurer}: {len(pred)} rows (expected {len(df_test_limited_casco)})")

print("\nMTPL predictions:")
for insurer, pred in preds_mtpl.items():
    print(f"  {insurer}: {len(pred)} rows (expected {len(df_test_mtpl)})")

Casco predictions:
  Insurer_A_price: 51616 rows (expected 51616)
  Insurer_B_price: 51616 rows (expected 51616)
  Insurer_C_price: 51616 rows (expected 51616)
  Insurer_D_price: 51616 rows (expected 51616)
  Insurer_E_price: 51616 rows (expected 51616)
  Insurer_F_price: 51616 rows (expected 51616)
  Insurer_G_price: 51616 rows (expected 51616)
  Insurer_H_price: 51616 rows (expected 51616)
  Insurer_I_price: 51616 rows (expected 51616)
  Insurer_J_price: 51616 rows (expected 51616)
  Insurer_K_price: 51616 rows (expected 51616)

Limited casco predictions:
  Insurer_A_price: 71589 rows (expected 71589)
  Insurer_B_price: 71589 rows (expected 71589)
  Insurer_C_price: 71589 rows (expected 71589)
  Insurer_D_price: 71589 rows (expected 71589)
  Insurer_E_price: 71589 rows (expected 71589)
  Insurer_F_price: 71589 rows (expected 71589)
  Insurer_G_price: 71589 rows (expected 71589)
  Insurer_H_price: 71589 rows (expected 71589)
  Insurer_I_price: 71589 rows (expected 71589)
  Insurer_J_p

In [46]:
# Get quote_ids from raw test file
quote_ids = df_test_raw['quote_id'].values

insurer_cols = [
    'Insurer_A_price', 'Insurer_B_price', 'Insurer_C_price',
    'Insurer_D_price', 'Insurer_E_price', 'Insurer_F_price',
    'Insurer_G_price', 'Insurer_H_price', 'Insurer_I_price',
    'Insurer_J_price', 'Insurer_K_price'
]

# Build submission
submission = pd.DataFrame({'quote_id': quote_ids})
for col in insurer_cols:
    submission[col] = 0.0

# Fill predictions per coverage split
for coverage, preds in [
    ('casco',         preds_casco),
    ('limited_casco', preds_limited),
    ('mtpl',          preds_mtpl),
]:
    idx = df_test_raw[df_test_raw['coverage'] == coverage].index
    for insurer in insurer_cols:
        if insurer in preds:
            submission.loc[idx, insurer] = preds[insurer]

# Verify
print(f"Submission shape: {submission.shape}")
print(f"Any nulls: {submission.isnull().sum().sum()}")
print(f"Any zeros: {(submission[insurer_cols] == 0).sum().sum()}")
print(f"\nSample:")
print(submission.head(3).to_string())

# Save with correct format
submission.to_csv('submission.csv', sep=';', decimal='.', index=False)
print("\nSaved to submission.csv")

# Verify file format
check = pd.read_csv('submission.csv', sep=';')
print(f"\nVerification:")
print(f"Shape: {check.shape}")
print(f"Columns: {check.columns.tolist()}")
print(check.head(3).to_string())

Submission shape: (164092, 12)
Any nulls: 0
Any zeros: 0

Sample:
   quote_id  Insurer_A_price  Insurer_B_price  Insurer_C_price  Insurer_D_price  Insurer_E_price  Insurer_F_price  Insurer_G_price  Insurer_H_price  Insurer_I_price  Insurer_J_price  Insurer_K_price
0    541293        73.973576       109.699019        83.072877       110.541282       137.318678        75.973214        80.042106        97.896092       152.348007       198.839255       164.735663
1    541294        30.387517        35.101336        31.992339        35.814282        43.654786        37.630029        31.230976        69.462252        32.605420        44.462241        49.509800
2    541295        25.912147        33.128008        30.449078        28.294381        45.383679        29.850049        35.108015        32.590334        27.596956        38.379017        33.066887

Saved to submission.csv

Verification:
Shape: (164092, 12)
Columns: ['quote_id', 'Insurer_A_price', 'Insurer_B_price', 'Insurer_C_price',

In [47]:
import os

# Round to 2 decimal places
submission[insurer_cols] = submission[insurer_cols].round(2)

# Save
submission.to_csv('submission.csv', sep=';', decimal='.', index=False)

size_mb = os.path.getsize('submission.csv') / (1024 * 1024)
print(f"File size: {size_mb:.2f} MB")
print(f"Rows: {len(submission)}")
print(f"\nSample:")
print(submission.head(3).to_string())

File size: 11.94 MB
Rows: 164092

Sample:
   quote_id  Insurer_A_price  Insurer_B_price  Insurer_C_price  Insurer_D_price  Insurer_E_price  Insurer_F_price  Insurer_G_price  Insurer_H_price  Insurer_I_price  Insurer_J_price  Insurer_K_price
0    541293            73.97           109.70            83.07           110.54           137.32            75.97            80.04            97.90           152.35           198.84           164.74
1    541294            30.39            35.10            31.99            35.81            43.65            37.63            31.23            69.46            32.61            44.46            49.51
2    541295            25.91            33.13            30.45            28.29            45.38            29.85            35.11            32.59            27.60            38.38            33.07
